# Research prototype — "does this idea even work?"

**AIMLCZG546 · Software Engineering for Machine Learning · Assignment II · Group 049**

This is the *research* half of Task 2 (research code vs production code).

It is how the conversion model actually started: one notebook, top to bottom, written to
answer a single question — *can we predict whether a browsing session ends in a purchase?*

It is intentionally left messy. **Do not fix it.** Its job is to be the "before" picture.
The production version of the same idea lives in the `ecom/` package.

In [1]:
import os, sys, json
import pandas as pd
import numpy   as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import *

In [2]:
df = pd.read_csv( "../data/sessions_train.csv" )
df.head()

,session_id,pages_viewed,session_duration_min,cart_items,avg_item_price,is_returning_user,discount_applied,device,converted
0,s000000,9,12.20,2,723.48,0,0,desktop,1
1,s000001,7,9.17,0,57.89,0,0,desktop,0
2,s000002,9,10.73,1,468.53,0,1,mobile,1
3,s000003,7,6.27,0,181.62,0,0,mobile,0
4,s000004,7,4.27,1,1549.23,1,0,desktop,1


In [3]:
df.shape, df['converted'].mean()

((5000, 9), np.float64(0.4606))

In [4]:
# quick and dirty cleanup
df['avg_item_price'] = df['avg_item_price'].fillna(0)
df["pages_per_minute"]=df["pages_viewed"]/(df["session_duration_min"]+1)
df["cart_value"] = df["cart_items"]*df["avg_item_price"]
df["has_cart"]=(df["cart_items"]>0).astype(int)
d = pd.get_dummies(df["device"])
df = pd.concat([df,d],axis=1)
df.head(3)

,session_id,pages_viewed,session_duration_min,cart_items,avg_item_price,is_returning_user,discount_applied,device,converted,pages_per_minute,cart_value,has_cart,desktop,mobile,tablet
0,s000000,9,12.20,2,723.48,0,0,desktop,1,0.681818,1446.96,1,True,False,False
1,s000001,7,9.17,0,57.89,0,0,desktop,0,0.688299,0.00,0,True,False,False
2,s000002,9,10.73,1,468.53,0,1,mobile,1,0.767263,468.53,1,False,True,False


In [5]:
X = df[["pages_viewed","session_duration_min","cart_items","avg_item_price","is_returning_user","discount_applied","pages_per_minute","cart_value","has_cart","desktop","mobile","tablet"]]
y = df["converted"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)   # no seed -> different numbers every run
m = LogisticRegression(max_iter=500)
m.fit(X_train,y_train)   # note: forgot to scale here, but scaled it in another cell earlier - which one is right?
p = m.predict(X_test)
print("accuracy", accuracy_score(y_test,p))
print("f1", f1_score(y_test,p))

accuracy 0.718
f1 0.6880530973451328


C:\Users\ravir\AppData\Local\Programs\Python\Python312-arm64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [6]:
# run it again - different answer, because there is no random seed
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)
m2 = LogisticRegression(max_iter=500)
m2.fit(X_train,y_train)
print("accuracy", accuracy_score(y_test,m2.predict(X_test)))

accuracy 0.731


C:\Users\ravir\AppData\Local\Programs\Python\Python312-arm64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
# tried a forest too, left it here half finished
#from sklearn.ensemble import RandomForestClassifier
#m3 = RandomForestClassifier()
#m3.fit(X_train,y_train)

In [8]:
# scoring a brand new session - feature order copy-pasted by hand, very easy to get wrong
new = [[12, 8.5, 2, 799.0, 1, 0, 1.26, 1598.0, 1, 0, 1, 0]]
print("prediction:", m.predict(new))

prediction: [1]


C:\Users\ravir\AppData\Local\Programs\Python\Python312-arm64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


## What is wrong with this notebook

Nothing here is *incorrect* — it is just not something anyone can depend on.

| Problem | Why it hurts in production |
| --- | --- |
| Hardcoded file path | Only runs on this machine, from this folder |
| No random seed | Two runs give two different accuracies — no reproducibility |
| Cleaning logic lives in the notebook | Serving code must copy-paste it and *will* drift |
| Feature list typed out by hand | One reordered column silently corrupts every prediction |
| Scaling applied in one cell but not another | Nobody knows which result is the real one |
| `from sklearn.metrics import *` | Unclear where names come from |
| `print()` instead of logging | No log levels, nothing captured on a server |
| No validation, no error handling | A malformed CSV fails deep inside sklearn with a cryptic error |
| Dead commented-out code | Reader cannot tell what is live |
| No tests | Nothing stops a regression |

➡️ The `ecom/` package fixes every row of this table. See `049.ipynb`, Task 2.